# GRPO Training — Job Scheduler Env (Colab)

Trains `Qwen/Qwen2.5-1.5B-Instruct` with Unsloth + GRPO on the Job Scheduler environment.

**Before running:** Make sure the runtime is set to **GPU** (Runtime → Change runtime type → T4 GPU).

In [ ]:
# Verify GPU is available
import torch
assert torch.cuda.is_available(), "No GPU found — change runtime to GPU first."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Install Dependencies

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy; get_numpy = f"numpy=={numpy.__version__}"
    except: get_numpy = "numpy"
    !uv pip install -qqq \
        
        "torch>=2.8.0" "triton>=3.4.0" {get_numpy} torchvision bitsandbytes "transformers==4.56.2" trackio \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth trackio
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

In [ ]:
# Remaining dependencies (openenv-core must be installed before starting the server)
!uv pip install -qqq trl datasets pydantic fastapi uvicorn "openenv-core[core]" httpx websockets

## 2. Clone the Repo

Replace the URL below with your repository URL, or skip this cell and upload the project files manually.

In [ ]:
import os

REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # <-- fill in your repo URL
REPO_DIR = "Job_Scheduler_Env"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"'{REPO_DIR}' already exists, skipping clone.")

# Change into the project directory so all imports resolve correctly
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Add the inner package dir to Python path so 'client' and 'models' are importable
import sys

pkg_dir = "/kaggle/working/Job_Scheduler_Env/Job_Scheduler_Env"
if pkg_dir not in sys.path:
    sys.path.insert(0, pkg_dir)
print("sys.path[0]:", sys.path[0])

## 3. Logging Setup

In [ ]:
import logging
import sys

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(levelname)-8s %(name)s | %(message)s",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler("train_debug.log", mode="w"),
    ],
)

for _noisy in ["unsloth", "unsloth_zoo", "transformers", "datasets", "trl", "peft", "accelerate", "torch"]:
    logging.getLogger(_noisy).setLevel(logging.ERROR)

logger = logging.getLogger(__name__)
logger.info("Logging configured.")

## 4. Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 768
lora_rank = 16  # increased from 4 — more capacity for learning the scheduling policy

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-1.5B-Instruct",
    load_in_4bit=True,
    max_seq_length=max_seq_length,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=lora_rank * 2,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

logger.info("Model loaded.")

## 5. Start the Environment Server

In [ ]:
import subprocess
import time
import urllib.request
from pathlib import Path

SERVER_ROOT = Path("/kaggle/working/Job_Scheduler_Env/Job_Scheduler_Env")

def _start_server():
    env = {**os.environ, "PYTHONPATH": str(SERVER_ROOT)}

    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "server.app:app", "--port", "8000"],
        cwd=str(SERVER_ROOT),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    for _ in range(30):
        try:
            urllib.request.urlopen("http://localhost:8000/docs")
            logger.info("Server is up.")
            return proc
        except Exception:
            time.sleep(1)

    stdout, stderr = proc.communicate(timeout=5)
    proc.kill()
    print("=== SERVER STDOUT ===")
    print(stdout.decode(errors="replace"))
    print("=== SERVER STDERR ===")
    print(stderr.decode(errors="replace"))
    raise RuntimeError("Server failed to start within 30 seconds — see output above.")


server = _start_server()
print("Server started.")

## 6. Connect Environment Client

In [ ]:
import sys

_pkg = "/kaggle/working/Job_Scheduler_Env/Job_Scheduler_Env"
if _pkg not in sys.path:
    sys.path.insert(0, _pkg)

from client import JobSchedulerEnvEnv
from models import JobSchedulerEnvAction

env = JobSchedulerEnvEnv(base_url="http://localhost:8000").sync()
logger.info("Environment client connected.")

## 7. Prompt Builder

In [ ]:
import json


def _build_prompt(obs) -> str:
    return (
        "Assign each job to a machine.\n\n"
        f"Jobs JSON:\n{json.dumps(obs.job_info)}\n\n"
        f"Machines JSON:\n{json.dumps(obs.machine_info)}\n\n"
        "Output one assignment per job in format:\n(job_id, machine_id)"
    )


def _sample_prompt() -> str:
    result = env.reset()
    return _build_prompt(result.observation)

## 8. Episode Runner & Reward Function

In [ ]:
import re


def _run_episode(action_strs: list[str]) -> float:
    result = env.reset()
    total_reward = 0.0

    for action_str in action_strs:
        if result.done:
            break
        try:
            result = env.step(JobSchedulerEnvAction(action=action_str))
            total_reward += float(result.reward or 0.0)
        except Exception:
            total_reward -= 0.5

    return total_reward


def env_reward(completions, **kwargs):
    scores = []

    for completion in completions:
        text = completion[0]["content"]
        matches = re.findall(r'\(\s*(\d+)\s*,\s*(\d+)\s*\)', text)

        if not matches:
            scores.append(-2.0)
            continue

        actions = [f"({a},{b})" for a, b in matches]

        try:
            reward = _run_episode(actions)
            scores.append(reward)
        except Exception:
            scores.append(-1.0)

    return scores

## 9. Build Dataset

In [ ]:
from datasets import Dataset
import random as _random

print("Sampling prompts across task levels...")

def _sample_prompt_level(level: int) -> str:
    result = env.reset(task_level=level)
    return _build_prompt(result.observation)

prompts = []
for _ in range(200):
    prompts.append(_sample_prompt_level(1))  # 3 jobs, 3 machines
for _ in range(200):
    prompts.append(_sample_prompt_level(2))  # 5 jobs, 4 machines
for _ in range(100):
    prompts.append(_sample_prompt_level(3))  # 7 jobs, 5 machines

_random.shuffle(prompts)

dataset = Dataset.from_list([
    {
        "prompt": [
            {
                "role": "system",
                "content": "You are a job scheduling AI. Always assign ALL jobs to machines using given data."
            },
            {
                "role": "user",
                "content": p
            }
        ],
        "answer": 0
    }
    for p in prompts
])

print(f"Dataset size: {len(dataset)} samples (200 lvl-1 + 200 lvl-2 + 100 lvl-3, shuffled)")

In [ ]:
# Debug: inspect the chat template
print("===== DEBUG CHAT TEMPLATE =====")
print(tokenizer.apply_chat_template(
    dataset[0]["prompt"],
    tokenize=False,
    add_generation_prompt=True
))
print("===============================")

In [ ]:
# Compute token lengths to set max_prompt_length
token_lens = [
    len(tokenizer.apply_chat_template(
        d["prompt"],
        tokenize=True,
        add_generation_prompt=True
    ))
    for d in dataset
]

max_prompt_length = max(token_lens) + 10
max_completion_length = max_seq_length - max_prompt_length

print(f"max_prompt_length   : {max_prompt_length}")
print(f"max_completion_length: {max_completion_length}")

## 10. Configure & Run GRPO Training

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    num_generations=4,            # 8 was too slow on T4; 4 still gives good contrast
    max_prompt_length=max_prompt_length,
    max_completion_length=max_completion_length,
    max_steps=500,
    logging_steps=10,
    report_to="none",
    output_dir="outputs_scheduler",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[env_reward],
    args=training_args,
    train_dataset=dataset,
)

In [ ]:
try:
    trainer.train()
    model.save_pretrained_merged(
        "outputs_scheduler/final_model",
        tokenizer,
        save_method="merged_16bit"
    )
    print("Training complete. Model saved to outputs_scheduler/final_model")
finally:
    env.close()
    server.terminate()
    server.wait(timeout=10)
    print("Server shut down.")

## 11. (Optional) Save Model to Google Drive

## 12. Compare Base Model vs Fine-Tuned Model

In [ ]:
import torch
import statistics

N_EVAL = 20  # number of episodes to evaluate each model

SYSTEM_MSG = "You are a job scheduling AI. Always assign ALL jobs to machines using given data."

def _generate(mdl, tok, prompt_text: str, max_new_tokens: int = 256) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user",   "content": prompt_text},
    ]
    input_ids = tok.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(mdl.device)

    with torch.no_grad():
        output_ids = mdl.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.eos_token_id,
        )

    new_tokens = output_ids[0][input_ids.shape[-1]:]
    return tok.decode(new_tokens, skip_special_tokens=True)


def _eval_model(mdl, tok, n=N_EVAL):
    rewards = []
    for _ in range(n):
        prompt = _sample_prompt()
        response = _generate(mdl, tok, prompt)
        matches = re.findall(r'\(\s*(\d+)\s*,\s*(\d+)\s*\)', response)
        if not matches:
            rewards.append(-2.0)
            continue
        actions = [f"({a},{b})" for a, b in matches]
        try:
            rewards.append(_run_episode(actions))
        except Exception:
            rewards.append(-1.0)
    return rewards


# ── Fine-tuned model (already in memory as `model`) ──────────────────────────
print(f"Evaluating fine-tuned model over {N_EVAL} episodes...")
FastLanguageModel.for_inference(model)
ft_rewards = _eval_model(model, tokenizer)

# ── Base model (fresh load, no LoRA) ─────────────────────────────────────────
print(f"Loading base model for comparison...")
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-1.5B-Instruct",
    load_in_4bit=True,
    max_seq_length=max_seq_length,
)
FastLanguageModel.for_inference(base_model)
print(f"Evaluating base model over {N_EVAL} episodes...")
base_rewards = _eval_model(base_model, base_tokenizer)

# ── Results ───────────────────────────────────────────────────────────────────
print("\n" + "="*45)
print(f"{'':20s} {'Base':>10s}  {'Fine-tuned':>10s}")
print("="*45)

print(f"{'Mean reward':20s} {statistics.mean(base_rewards):>10.3f}  {statistics.mean(ft_rewards):>10.3f}")
print(f"{'Median reward':20s} {statistics.median(base_rewards):>10.3f}  {statistics.median(ft_rewards):>10.3f}")
print(f"{'Std dev':20s} {statistics.stdev(base_rewards):>10.3f}  {statistics.stdev(ft_rewards):>10.3f}")
print(f"{'Min reward':20s} {min(base_rewards):>10.3f}  {min(ft_rewards):>10.3f}")
print(f"{'Max reward':20s} {max(base_rewards):>10.3f}  {max(ft_rewards):>10.3f}")
print("="*45)

delta = statistics.mean(ft_rewards) - statistics.mean(base_rewards)
print(f"\nMean reward delta: {delta:+.3f} ({'improvement' if delta > 0 else 'regression'})")

In [ ]:
# Uncomment to mount Drive and copy the saved model
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r outputs_scheduler/final_model /content/drive/MyDrive/job_scheduler_model